In [2]:
!uv pip install fg-data-profiling
###!uv pip install ipywidgets

Using Python 3.13.15 environment at: /workspaces/aml-working-paper/.venv
⠋ wrapt==2.5.0                                                                  Resolved 55 packages in 3.59s
░░░░░░░░░░░░░░░░░░░░ [0/2] Installing wheels...                                 warning: Failed to hardlink files; falling back to full copy. This may lead to degraded performance.
         If the cache and target directories are on different filesystems, hardlinking may not be supported.
         If this is intentional, set `export UV_LINK_MODE=copy` or use `--link-mode=copy` to suppress this warning.
Installed 2 packages in 1.74s0.0                            
 + fg-data-profiling==4.20.0
 + shapely==2.2.0


# Creating the Synthetic Dataset

### 1. Import packages

In [3]:
###------Installed packages-----
import numpy as np
import pandas as pd
from collections import Counter
from data_profiling import ProfileReport
from scipy.special import softmax, expit

from aml_working_paper.simulation.states import sample_next_state

###------Working Paper's functions------
from aml_working_paper.simulation.equations import simulate_accounts, generate_economic_context, generate_account_exposure,generate_structuring_score, generate_cross_border_score,generate_transaction_volume
from aml_working_paper.config import RAW_DATA_DIR, FIGURES_DIR

### 2. DGP structure

In [4]:
#------ define some important variables

### Start from an initial point
initial_economic_context=0.0
economic_context_path=0
initial_aml_state=0

### replicate results
seed=42 
seed_sequence = np.random.SeedSequence(seed) 

### How many repetitions
periods = 100

### Values to calculate economic context
rho=0.8 ### it means how much the current economic context stays/lasts between periods
sigma=0.5 ### the 

aml_state = 1

### 3. Define equations (equations.py)

##### 3.1 **Economic context**:  Generate a simple economic context using an AR(1) process (autoregressive. so it depends on the previous value). 0 No change in the economy's situation. 1 Change in the economy's situation. Shock is introduced to add randomness. Future work could include inflation, tourism or other exogenous variables. 

In [5]:
def generate_economic_context(economic_context, rng, rho=0.8, sigma=0.5):
    shock = rng.normal(0, sigma)
    economic_context = float(rho * economic_context + shock)
    return economic_context

In [6]:
### for economic context, a different random secuence is generated by using one of the already generate child seeds
### this is done so each generator derivates from the same seed but it doesn't depend on that seed
### this gives more randomness and simulates shocks too. 
if economic_context_path == 0:
    child_seeds = seed_sequence.spawn(6)
    rng_economic = np.random.default_rng(child_seeds[0])
    start = 1
else:
    child_seeds = seed_sequence.spawn(5)
    start = 0

In [7]:
rng_economic = np.random.default_rng(child_seeds[0])

In [8]:
economic_context = generate_economic_context(initial_economic_context, rng_economic)

In [9]:
economic_context

0.20916498266218878

##### 3.1.1 **Simulate Economic context**: The economic context has to be shared across accounts at each time period. This is why simulate_economic_context() function exists:

In [10]:
rng = np.random.default_rng(seed)
context = initial_economic_context
values = []
for _ in range(periods):
    shock = rng.normal(0, sigma)
    economic_context = float(rho * economic_context + shock) ### this is simulate_economic_context()
    values.append(economic_context)
values = np.array(values)

In [11]:
values

array([ 0.31969053, -0.26423963,  0.16383389,  0.60134947, -0.49443802,
       -1.04664017, -0.77339193, -0.77683484, -0.62986845, -0.93041673,
       -0.30463439,  0.14518845,  0.14916611,  0.68295349,  0.78011747,
        0.19444774,  0.33993358, -0.20749443,  0.2732296 ,  0.19362073,
        0.0624654 , -0.29049245,  0.37887671,  0.22583663, -0.03349461,
       -0.20286246,  0.10386462,  0.26581373,  0.41901729,  0.55062433,
        1.51132327,  1.00585111,  0.54855952,  0.03196125,  0.33355871,
        0.83133312,  0.60809276,  0.06639597, -0.35912383,  0.03799733,
        0.40202495,  0.59319709,  0.14180282,  0.22952292,  0.24196124,
        0.30291329,  0.67804502,  0.65423379,  0.86284381,  0.72406459,
        0.72381137,  0.89469321, -0.01282334, -0.17009428, -0.37126175,
       -0.61644833, -0.63072979,  0.24288683, -0.2386061 ,  0.2932543 ,
       -0.60683145, -0.65290767, -0.44094961, -0.05964852,  0.30789448,
        0.6429892 ,  0.34002882,  0.04084716,  0.46166567,  0.27

##### 3.2 **Account Exposure**: Exposure represents the level that an account is vulnerable to risky accounts/networks. The idea behind this function is that economic context influeces an account exposition but it doesn't entirely decide the exposure itself. Two accounts witht the same economic context can have different exposure.

In [12]:
### Use the same child_seed to have the influence of the calculated economic context
rng_network = np.random.default_rng(child_seeds[start])

In [13]:
random_noise = rng_network.normal(0, 0.7)
### -2 was selected so, when economic_context and random_noise is 0 or really low, the exposition is low too. 0.8 is to make the exposure rise/decrease whit the economic context
exposure = -2 + 0.8 * economic_context + random_noise
### expit (sigmoid) is used to convert any number to a value between 0 and 1
account_exposure = float(expit(exposure))

In [14]:
account_exposure

0.08727368935999792

#### 3.3 **generate_structuring_score** : defined to simulate the practice of breaking down large transactions into smaller ones. It returns a values between 0 and 1. It also uses the dependent variable aml state in order to give the HMM model a small clue of the relationship between the target variable and the ones it is analyzing. Nonetheless, noise is added to not make the relationship obvious.

In [15]:
noise = rng.normal(0, 0.6)
### When aml state is 0, -2 becomes a smaller number when passes through sigmoid. -2 and 1.3 are arbitrary values 
structuring = -2 + 1.3 * aml_state + noise ### aml state is 0 as an example. 
### sigmoid is used to avoid negative values and giving accounts with lower aml states a higher structuring score
structuring_score = float(expit(structuring))
structuring_score

0.28355456159540837

#### 3.4 **generate_cross_border_score**: it tries to simulate the poproportion of an account's activity with other ones. Both explanatory variables have positive coefficients, reflecting our assumption that higher network exposure and higher AML states are associated with greater cross-border activity.

In [16]:
noise = rng.normal(0, 0.6)
### -2.5, 1.5 and 0.8 are arbitrary values. Yet, they follow the same logic as generate_structuring_score and generate_account_exposure
linear_predictor = (-2.5 + 1.5 * account_exposure + 0.8 * aml_state + noise)
### sigmoid is used again to avoid negative values and to avoid the same issue as generate_structuring_score
cross_border_ratio_score = float(expit(linear_predictor))

#### 3.5 **generate_transaction_volume**: It generates the volume of transactions for an account. It uses a lognormal distribution whose log-scale mean depends on economic context, latent AML state, and structuring behavior.

In [17]:
### 2.5, 0.25, 0.20 and 0.60 are arbitrary values.
log_mean = (2.5 + 0.25 * economic_context + 0.20 * aml_state + 0.60 * structuring_score)
### Lognormal is used so the distribution of the transactions is positive, to the right and with many transactions accumulated in the beginning.  
### Lognormal also adds noise
volume = float(rng.lognormal(log_mean, 0.3))
volume

10.807868738898225

### 4. Simulate an account's behavior (equations.py and state.py)

#### 4.1 Sample next state (state.py)

In [18]:
### Create a simple base where probabilities can be calculated from
### More information about this on states.py
base_transition = np.array(
    [
    [0.94, 0.05, 0.01],
    [0.10, 0.75, 0.15],
    [0.03, 0.17, 0.80],
    ],
    dtype=float,
)

### Create an account exposure effect array. 
### This helps to modify the probabilities based on the account's exposure. 
### For this working paper, account exposure affects directly the change in aml state
### all the other equations work as context for the account's transactions. 
### But! for future work other effects could be added.  
account_exposure_effect = np.array([-1.0, 0.25, 1.25], dtype=float)

In [19]:
base = base_transition[initial_aml_state]
### Adjust the the base and network exposure by penalizing the highest aml states
adjusted = (np.log(base) + account_exposure * account_exposure_effect)
### softmax so values are between 0 and 1
probabilities = softmax(adjusted)

In [20]:
next_aml_state = int(rng.choice(3, p=probabilities))

In [21]:
next_aml_state

0

#### 4.2 Simulate one account

In [22]:
### Generate different pseudo random from a random seed. This is done for reproducibility.
seed_sequence = np.random.SeedSequence(seed)
### Get 6 random seeds for the five first generate functions and the sample_next_state.
if economic_context_path is None:
    child_seeds = seed_sequence.spawn(6)
    ### for economic context, a different random secuence is generated by using one of the already generate child seeds
    ### this is done so each generator is created from the same seed but it doesn't depend on that specific seed
    ### this gives more randomness and simulates shocks too. 
    rng_economic = np.random.default_rng(child_seeds[0])
    start = 1
else:
    child_seeds = seed_sequence.spawn(5)
    start = 0

###create random generated numbers for all equathttps://plotly.com/examples/ions. 
### This gives more independence and control when experimenting
### If, for example, structuring score equations is going to be experimented on, the random number will only change that equation distribution
### it won't change the other equations results. 
rng_account = np.random.default_rng(child_seeds[start])
rng_state = np.random.default_rng(child_seeds[start + 1])
rng_structuring = np.random.default_rng(child_seeds[start + 2])
rng_cross_border = np.random.default_rng(child_seeds[start + 3])
rng_volume = np.random.default_rng(child_seeds[start + 4])

economic_context = initial_economic_context
previous_aml_state = initial_aml_state
rows = []

### Iterate for all the periods of the account and generate the different context whilesaving the previous ones
for t in range(periods):
    ### if there is no economic context (first period) create it, after that use the current one. 
    if economic_context_path is None:
        economic_context = generate_economic_context(economic_context, rng_economic)
    #else:
    #    print(economic_context_path)

    account_exposure = generate_account_exposure(economic_context, rng_account)
    aml_state = sample_next_state(previous_state=previous_aml_state, account_exposure=account_exposure, rng=rng_state)
    structuring_score = generate_structuring_score(aml_state, rng_structuring)
    cross_border_ratio = generate_cross_border_score(account_exposure, aml_state, rng_cross_border)
    transaction_volume = generate_transaction_volume(economic_context, aml_state, structuring_score, rng_volume)

    ### save all the results into a list
    rows.append({
        "time": t,
        "economic_context": economic_context,
        "account_exposure": account_exposure,
        "previous_aml_state": previous_aml_state,
        "aml_state": aml_state,
        "structuring_score": structuring_score,
        "cross_border_ratio": cross_border_ratio,
        "transaction_volume": transaction_volume,
    })

    previous_aml_state = aml_state

complete_account = pd.DataFrame(rows)

In [23]:
### The result is the behavior of one account for 100 periods (periods = 100 on section 2)
complete_account

,time,economic_context,account_exposure,previous_aml_state,aml_state,structuring_score,cross_border_ratio,transaction_volume
0,0,0.0,0.153531,0,0,0.047896,0.088672,11.310681
1,1,0.0,0.171349,0,0,0.239322,0.203120,8.545814
2,2,0.0,0.121335,0,0,0.172799,0.036722,12.037491
3,3,0.0,0.059582,0,0,0.076038,0.083870,18.884623
4,4,0.0,0.273876,0,1,0.398989,0.141301,19.926959
...,...,...,...,...,...,...,...,...
95,95,0.0,0.084367,2,1,0.148267,0.238731,12.234173
96,96,0.0,0.121569,1,1,0.176761,0.155514,23.084455
97,97,0.0,0.074394,1,1,0.336616,0.222137,15.949724
98,98,0.0,0.252377,1,1,0.364460,0.113585,18.240825


### 5. Simulate 100 accounts for 100 periods

##### Simulate accounts only iterates the previous blocks of code for a number of accounts set. In this case, using the function summarizes all of the code above

In [ ]:
df_aml = simulate_accounts(n_accounts=100, periods=100, seed=42)

0.5294191206471622
0.5509348129301915
0.5270929273182019
-0.9911294715981614
-0.8799044692253528
-0.8471605943804087
-0.1387837562530596
-0.1133986811165942
0.14616631482264572
0.3441128876558136
1.0454455605977562
1.8632197366276286
1.4090397681064275
1.0335720830924324
1.16547188277751
0.6008388308611561
0.06781168782467528
-0.3667237378200051
-0.9854339683493745
-0.7126671174813649
-0.030767724186195777
-0.7093349317295239
-0.3445571659877848
0.3580355275683281
0.6996062137625099
0.7905721801426416
0.10049515820730648
-0.29271534455396
-0.08171878935746787
-0.22170508409966405
-0.6880684140268958
-0.5368173134467406
-1.7069945388189196
-2.7343948063157635
-2.6300626572106642
-2.31287081487583
-2.308490015432968
-1.2083426173735776
-1.2393961357935583
-0.758471191719842
-0.6905287864676799
-1.0889988260132923
-0.47245792436936845
-0.16014600896326325
-0.4129031562981994
-0.5020246459634318
-0.15669547460999494
0.013187110649954759
-0.05450259150876058
0.2205371182797156
-0.1595989565

In [25]:
df_aml.head()

,account_id,time,economic_context,account_exposure,previous_aml_state,aml_state,structuring_score,cross_border_ratio,transaction_volume
0,0,0,0.529419,0.062469,0,0,0.097453,0.060671,14.753837
1,0,1,0.550935,0.183982,0,0,0.257241,0.058192,17.693912
2,0,2,0.527093,0.247254,0,0,0.069465,0.171748,13.516781
3,0,3,-0.991129,0.117232,0,0,0.247049,0.077314,10.770161
4,0,4,-0.879904,0.025602,0,0,0.078729,0.062309,10.318240


In [26]:
df_aml.shape

(10000, 9)

### 6. Save the sythetic dataset

In [ ]:
df_aml.to_csv("{}/synthetic_df_aml.csv".format(RAW_DATA_DIR), index=False)

### 7. Generate a small report from the created synthetic data

In [ ]:
profile = ProfileReport(df_aml,
    title="EDA AML synthetic dataset",
    correlations={
        "auto": {"calculate": True},
        "pearson": {"calculate": True},
        "spearman": {"calculate": True},
    },
)

profile.to_file("{}/dgp_eda.html".format(FIGURES_DIR))

In [ ]:
profile.to_notebook_iframe()